# Dependencies

In [ ]:
from math import erfc, sqrt
from numpy import set_printoptions
from scipy.special import gammaincc
from tabulate import tabulate
import random
import matplotlib.pyplot as plt

# Frequency test

In [15]:
def countOnesList(L):
    """
    Count the number of extra ones of a given list of bits.
    
    :param L: a list of 1s and 0s.
    :return: the number of extra 1s compared to 0s (negative values admitted).
            countOnes([0,1,0,1]) = 0 (no extra ones)
            countOnes([0,1,1,1]) = 2 (two extra ones)
            countOnes([0,1,0,0]) = -2 (two extra zeros)
    """
    Ones = [2*i - 1 for i in L]
    return sum(Ones)

def frequencyTest(L, n):
    """
    Evaluates the proportion of 1s with respect to 0s.

    :param L: a list of 1s and 0s
    :return: a p-value
    """
    extraOnes = countOnesList(L)
    stat = abs(extraOnes) / sqrt(n)
    z = stat/sqrt(2)
    pvalue = erfc(z)
    return pvalue

# Frequency test M-Block

In [13]:
def partList(L, M, n):
    """
    Partitions list L in blocks of size M
    """
    N = n // M
    R = [L[i*M:(i+1)*M] for i in range(N)]
    return R

def frequencyBlockTest(L, M, n):
    """
    Tests the distribution of ones of a given bit sequence L within each M-block.

    :param L: a list of ones and zeros,
    :param M: the size of each block
    
    :return: pvalue
    """
    N = n // M
    Blocks = partList(L, M, n)
    Ones = [b.count(1)/M for b in Blocks] #contains the proportion of ones within each block
    Terms = [(k - 1/2)**2 for k in Ones]
    chiSquared = 4 * M * sum(Terms)
    pvalue = gammaincc(N/2, chiSquared/2)
    return pvalue

# Runs test

In [25]:
def onesProportion(L, n):
    """
    Returns the proportion of 1s in a given list L of lenth n.
    """
    return L.count(1)/n

def freqOnesTest(L, n, t = 0.5):
    """
    returns True if the proportion of ones in a list L of length n is close to t (whose default is set to 0.5;
    False otherwise.
    """
    ones = onesProportion(L, n)
    return abs(ones - 1/2) < t

def countRuns(L):
    """
    Counts the number of runs within a list of zeros and ones.
    """
    c = 1
    for k in range(len(L) - 1):
        if L[k] != L[k + 1]:
            c += 1
    return c

def obStatistic(L, n):
    """
    Computes a statistic for the observable L.
    """
    p = onesProportion(L, n)
    v = countRuns(L)
    return abs(v - 2*n*p*(1-p)) / (2*sqrt(2*n)*p*(1-p))

def runsTest(L, n):
    """
    calculates a p-value for runs of L.
    """
    freqTest = freqOnesTest(L, n)
    if not freqTest:
        pvalue = 0
    else:
        statistic = obStatistic(L, n)
        pvalue = erfc(statistic)
    return pvalue

In [30]:
def testResults(L, n):
    """
    Given a List of bits it performs all former tests and returns for each test its pvalue.
    """
    M = 10
    Tests = ["Frequency test", "Frequency block test", "Runs"]
    freqPv = frequencyTest(L, n)
    freqBlockPv = frequencyBlockTest(L, M, n)
    runsPv = runsTest(L, n)
    Results = [freqPv, freqBlockPv, runsPv]
    Headers = ["Test", "p-value"]
    Values = [[Tests[i], Results[i]] for i in range(len(Tests))]
    print(tabulate(Values, headers=Headers, tablefmt="grid"))


def pVals(lBits, n, M):
    """
    :param lBits: a list of bits,
    :param n: the length of each bit,
    :param M: the size of each block,

    :return: a dictionary of "NameTest - pvalues"
    """
    tests = ["frequencies", "blocks", "runs"]
    freq = [frequencyTest(i, n) for i in lBits]
    freqBlock = [frequencyBlockTest(i, M, n) for i in lBits]
    runs = [runsTest(i, n) for i in lBits]
    return {tests[0]:freq, tests[1]:freqBlock, tests[2]:runs}

